# Transform Payments Data
1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
2. Map payment_status to contain descriptive vaules(1-Success, 2-Pending, 3-Cancelled, 4-Failed)
3. Write transformed data to the Silver schema

In [0]:
df = spark.table('gizmobox.bronze.payments')
display(df)

## 1. Extract Date and Time form payment_timestamp

[Documentation sql.functions.data_format](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html)

In [0]:
from pyspark.sql import functions as F

df_extrated_payments = (
    df
    .select(
        'payment_id',
        'order_id',
        F.date_format('payment_timestamp', 'yyyy-MM-dd').alias('payment_date'),
        F.date_format('payment_timestamp', 'HH:mm:ss').alias('payment_time'),
        'payment_status',
        'payment_method'
        )
    )

display(df_extrated_payments)

## 2. Map payment_status to contain descriptive values
(1-Success, 2-Pending, 3-Cancelled, 4-Failed)

[functions.when](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html?highlight=when#pyspark.sql.functions.when)

In [0]:
### with map

# from itertools import chain

# payment_status = {
#     "1": "Success",
#     "2": "Pending",
#     "3": "Cancelled",
#     "4": "Failed"
# }

# status_map = F.create_map([F.lit(x) for x in chain(*payment_status.items())])

# df_mapped_payments = (
#     df_extrated_payments
#     .select(
#         'payment_id',
#         'order_id',
#         'payment_date',
#         'payment_time',
#         status_map[F.col('payment_status').cast('string')].alias('payment_status'),
#         'payment_method'
#     )
# )

## With when
df_mapped_payments =(
    df_extrated_payments
    .select(
        'payment_id',
        'order_id',
        'payment_date',
        'payment_time',
        F.when(df_extrated_payments.payment_status == 1, 'Success')
        .when(df_extrated_payments.payment_status == 2, 'Pending')
        .when(df_extrated_payments.payment_status == 3, 'Cancelled')
        .when(df_extrated_payments.payment_status == 4, 'Failed')
        .otherwise('Unknown')
        .alias('payment_status'),
        'payment_method')
)



display(df_mapped_payments)

## 3. Write transformed data to the Silver schema

In [0]:
df_mapped_payments.writeTo('gizmobox.silver.py_payments').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_payments;